In [1]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np

BASE_DIR = "/content/drive/MyDrive/Mammogram_Project"

X = np.load(f"{BASE_DIR}/density_roi_x.npy")
Y = np.load(f"{BASE_DIR}/density_roi_y.npy")
D = np.load(f"{BASE_DIR}/density_labels.npy")

print("Images :", X.shape)
print("Masks  :", Y.shape)
print("Labels :", D.shape)

print("\nImage dtype:", X.dtype)
print("Mask dtype :", Y.dtype)

print("\nDensity classes:", np.unique(D, return_counts=True))

Mounted at /content/drive
Images : (1696, 224, 224)
Masks  : (1696, 224, 224)
Labels : (1696,)

Image dtype: uint8
Mask dtype : uint8

Density classes: (array([1, 2, 3, 4], dtype=uint8), array([337, 757, 449, 153]))


In [2]:
from sklearn.model_selection import train_test_split

# First split: 80% Train, 20% Temporary
X_train, X_temp, Y_train, Y_temp, D_train, D_temp = train_test_split(
    X,
    Y,
    D,
    test_size=0.20,
    random_state=42,
    stratify=D
)

# Second split: 10% Validation, 10% Test
X_val, X_test, Y_val, Y_test, D_val, D_test = train_test_split(
    X_temp,
    Y_temp,
    D_temp,
    test_size=0.50,
    random_state=42,
    stratify=D_temp
)

print("Train:", X_train.shape, Y_train.shape, D_train.shape)
print("Validation:", X_val.shape, Y_val.shape, D_val.shape)
print("Test:", X_test.shape, Y_test.shape, D_test.shape)

Train: (1356, 224, 224) (1356, 224, 224) (1356,)
Validation: (170, 224, 224) (170, 224, 224) (170,)
Test: (170, 224, 224) (170, 224, 224) (170,)


In [3]:
def show_distribution(name, labels):

    unique, counts = np.unique(labels, return_counts=True)

    print(f"\n{name}")

    for density, count in zip(unique, counts):
        print(f"Density {density}: {count}")


show_distribution("Train", D_train)
show_distribution("Validation", D_val)
show_distribution("Test", D_test)


Train
Density 1: 270
Density 2: 605
Density 3: 359
Density 4: 122

Validation
Density 1: 33
Density 2: 76
Density 3: 45
Density 4: 16

Test
Density 1: 34
Density 2: 76
Density 3: 45
Density 4: 15


In [4]:
# Normalize images to [0, 1]

X_train = X_train.astype(np.float32) / 255.0
X_val   = X_val.astype(np.float32) / 255.0
X_test  = X_test.astype(np.float32) / 255.0

# Convert masks to binary float

Y_train = (Y_train > 0).astype(np.float32)
Y_val   = (Y_val > 0).astype(np.float32)
Y_test  = (Y_test > 0).astype(np.float32)

print("Image range:", X_train.min(), X_train.max())
print("Mask values:", np.unique(Y_train))

Image range: 0.003921569 1.0
Mask values: [0. 1.]


In [5]:
X_train = np.repeat(X_train[..., None], 3, axis=-1)
X_val   = np.repeat(X_val[..., None], 3, axis=-1)
X_test  = np.repeat(X_test[..., None], 3, axis=-1)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: (1356, 224, 224, 3)
X_val: (170, 224, 224, 3)
X_test: (170, 224, 224, 3)


In [6]:
Y_train = np.expand_dims(Y_train, axis=1)
Y_val   = np.expand_dims(Y_val, axis=1)
Y_test  = np.expand_dims(Y_test, axis=1)

print("Y_train:", Y_train.shape)
print("Y_val:", Y_val.shape)
print("Y_test:", Y_test.shape)

Y_train: (1356, 1, 224, 224)
Y_val: (170, 1, 224, 224)
Y_test: (170, 1, 224, 224)


In [7]:
import torch
from torch.utils.data import Dataset, DataLoader


class MammogramDataset(Dataset):

    def __init__(self, images, masks, labels):

        self.images = images
        self.masks = masks
        self.labels = labels


    def __len__(self):

        return len(self.images)


    def __getitem__(self, idx):

        # Image: HWC -> CHW
        image = torch.tensor(
            self.images[idx].transpose(2, 0, 1),
            dtype=torch.float32
        )

        # Mask already CHW
        mask = torch.tensor(
            self.masks[idx],
            dtype=torch.float32
        )

        density = torch.tensor(
            self.labels[idx],
            dtype=torch.long
        )

        return image, mask, density

In [8]:
train_dataset = MammogramDataset(
    X_train,
    Y_train,
    D_train
)

val_dataset = MammogramDataset(
    X_val,
    Y_val,
    D_val
)

test_dataset = MammogramDataset(
    X_test,
    Y_test,
    D_test
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))

Train samples: 1356
Validation samples: 170
Test samples: 170


In [9]:
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

In [10]:
images, masks, density = next(iter(train_loader))

print("Images :", images.shape)
print("Masks  :", masks.shape)
print("Density:", density.shape)

Images : torch.Size([8, 3, 224, 224])
Masks  : torch.Size([8, 1, 224, 224])
Density: torch.Size([8])


In [11]:
!git clone https://github.com/HuCaoFighting/Swin-Unet.git

Cloning into 'Swin-Unet'...
remote: Enumerating objects: 130, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 130 (delta 37), reused 19 (delta 19), pack-reused 67 (from 2)
Receiving objects: 100% (130/130), 58.55 KiB | 8.36 MiB/s, done.
Resolving deltas: 100% (53/53), done.


In [12]:
%cd Swin-Unet

/content/Swin-Unet


In [13]:
!pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.3/156.3 kB 15.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 19.2 MB/s eta 0:00:00
  Created wheel for medpy: filename=MedPy-0.5.2-py3-none-any.whl size=224710 sha256=88f31b11737dde36ba150b6f913aad7f49b71e85ff4ed47b508792d2f07b4947
  Stored in directory: /root/.cache/pip/wheels/89/5a/f8/b3def53b9c2133d2f8698ea2173bb5df63bd8e761ce8e9aec9
Successfully built medpy


In [14]:
# !pip install timm yacs einops ml_collections

In [15]:
!mkdir pretrained_ckpt
!wget -P pretrained_ckpt https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth

--2026-07-17 16:15:58--  https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth
Resolving github.com (github.com)... 20.205.243.166
Connecting to github.com (github.com)|20.205.243.166|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/357198522/fd006b80-9bd3-11eb-8445-769d89efab4e?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-07-17T16%3A50%3A45Z&rscd=attachment%3B+filename%3Dswin_tiny_patch4_window7_224.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-07-17T15%3A50%3A31Z&ske=2026-07-17T16%3A50%3A45Z&sks=b&skv=2018-11-09&sig=NZCfScMUARV75JTadbwq1wYWqKk%2BJD43FBwYEAwc2EU%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4NDMwODU1OSwibmJmIjoxNzg0MzA0OTU5LC

In [16]:
import sys
sys.path.append('/content/Swin-Unet')

In [20]:
!pip install yacs
from networks.vision_transformer import SwinUnet
from config import get_config

In [31]:
import argparse

args = argparse.Namespace()

args.cfg = "configs/swin_tiny_patch4_window7_224_lite.yaml"

args.opts = []

args.batch_size = 8
args.zip = False
args.cache_mode = "part"
args.resume = ""
args.accumulation_steps = 0
args.use_checkpoint = False
args.amp_opt_level = "O1"
args.tag = ""
args.eval = False
args.throughput = False
args.local_rank = 0

config = get_config(args)

=> merge config from configs/swin_tiny_patch4_window7_224_lite.yaml


In [32]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SwinUnet(
    config,
    img_size=224,
    num_classes=1
)

model.load_from(config)

model = model.to(device)

SwinTransformerSys expand initial----depths:[2, 2, 2, 2];depths_decoder:[1, 2, 2, 2];drop_path_rate:0.2;num_classes:1
---final upsample expand_first---
pretrained_path:./pretrained_ckpt/swin_tiny_patch4_window7_224.pth
---start load pretrained modle of swin encoder---


In [33]:
images, masks, density = next(iter(train_loader))

images = images.to(device)

with torch.no_grad():

    outputs = model(images)

print(outputs.shape)

torch.Size([8, 1, 224, 224])


In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [42]:
import torch.nn as nn

class DiceLoss(nn.Module):

    def __init__(self, smooth=1):

        super().__init__()
        self.smooth = smooth

    def forward(self, pred, target):

        pred = torch.sigmoid(pred)

        pred = pred.view(-1)
        target = target.view(-1)

        intersection = (pred * target).sum()

        dice = (2. * intersection + self.smooth) / (
            pred.sum() + target.sum() + self.smooth
        )

        return 1 - dice

In [43]:
bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def criterion(pred, target):
    return bce_loss(pred, target) + dice_loss(pred, target)

In [44]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def calculate_metrics(pred, target):

    pred = (torch.sigmoid(pred) > 0.5).float()

    pred_np = pred.cpu().numpy().flatten()
    target_np = target.cpu().numpy().flatten()

    intersection = (pred_np * target_np).sum()

    dice = (2 * intersection + 1e-7) / (
        pred_np.sum() + target_np.sum() + 1e-7
    )

    iou = (intersection + 1e-7) / (
        pred_np.sum() + target_np.sum() - intersection + 1e-7
    )

    acc = accuracy_score(target_np, pred_np)
    precision = precision_score(target_np, pred_np, zero_division=0)
    recall = recall_score(target_np, pred_np, zero_division=0)
    f1 = f1_score(target_np, pred_np, zero_division=0)

    return dice, iou, acc, precision, recall, f1

In [45]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

In [46]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=3
)

In [47]:
scaler = torch.amp.GradScaler("cuda")

In [48]:
from tqdm import tqdm
import numpy as np
import torch
import os

EPOCHS = 20

history = {
    "train_loss": [],
    "val_loss": [],
    "train_dice": [],
    "val_dice": [],
    "train_iou": [],
    "val_iou": [],
    "train_acc": [],
    "val_acc": [],
    "train_precision": [],
    "val_precision": [],
    "train_recall": [],
    "val_recall": [],
    "train_f1": [],
    "val_f1": []
}

best_loss = np.inf
patience = 10
counter = 0

SAVE_PATH = "/content/drive/MyDrive/Mammogram_Project/swin_unet_best.pth"

for epoch in range(EPOCHS):

    #########################################################
    # Training
    #########################################################

    model.train()

    train_loss = 0

    train_dice = []
    train_iou = []
    train_acc = []
    train_precision = []
    train_recall = []
    train_f1 = []

    loop = tqdm(train_loader)

    for images, masks, _ in loop:

        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast(device_type="cuda"):

            outputs = model(images)

            loss = criterion(outputs, masks)

        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()

        train_loss += loss.item()

        dice, iou, acc, precision, recall, f1 = calculate_metrics(
            outputs,
            masks
        )

        train_dice.append(dice)
        train_iou.append(iou)
        train_acc.append(acc)
        train_precision.append(precision)
        train_recall.append(recall)
        train_f1.append(f1)

        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item())

    #########################################################
    # Validation
    #########################################################

    model.eval()

    val_loss = 0

    val_dice = []
    val_iou = []
    val_acc = []
    val_precision = []
    val_recall = []
    val_f1 = []

    with torch.no_grad():

        for images, masks, _ in val_loader:

            images = images.to(device)
            masks = masks.to(device)

            with torch.amp.autocast(device_type="cuda"):

                outputs = model(images)

                loss = criterion(outputs, masks)

            val_loss += loss.item()

            dice, iou, acc, precision, recall, f1 = calculate_metrics(
                outputs,
                masks
            )

            val_dice.append(dice)
            val_iou.append(iou)
            val_acc.append(acc)
            val_precision.append(precision)
            val_recall.append(recall)
            val_f1.append(f1)

    #########################################################
    # Average Metrics
    #########################################################

    train_loss /= len(train_loader)
    val_loss /= len(val_loader)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_dice"].append(np.mean(train_dice))
    history["val_dice"].append(np.mean(val_dice))

    history["train_iou"].append(np.mean(train_iou))
    history["val_iou"].append(np.mean(val_iou))

    history["train_acc"].append(np.mean(train_acc))
    history["val_acc"].append(np.mean(val_acc))

    history["train_precision"].append(np.mean(train_precision))
    history["val_precision"].append(np.mean(val_precision))

    history["train_recall"].append(np.mean(train_recall))
    history["val_recall"].append(np.mean(val_recall))

    history["train_f1"].append(np.mean(train_f1))
    history["val_f1"].append(np.mean(val_f1))

    #########################################################
    # Print Metrics
    #########################################################

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )

    print(
        f"Train Dice: {np.mean(train_dice):.4f} | "
        f"Val Dice: {np.mean(val_dice):.4f}"
    )

    print(
        f"Train IoU: {np.mean(train_iou):.4f} | "
        f"Val IoU: {np.mean(val_iou):.4f}"
    )

    #########################################################
    # Save Best Model
    #########################################################

    if val_loss < best_loss:

        best_loss = val_loss

        torch.save(model.state_dict(), SAVE_PATH)

        counter = 0

        print("Best model saved.")

    else:

        counter += 1

        print(f"EarlyStopping Counter: {counter}/{patience}")

    #########################################################
    # Early Stopping
    #########################################################

    if counter >= patience:

        print("Early stopping triggered.")

        break

Epoch [1/20]: 100%|██████████| 170/170 [00:36<00:00,  4.63it/s, loss=0.27]



Epoch 1/20
Train Loss: 0.5396 | Val Loss: 0.4598
Train Dice: 0.6487 | Val Dice: 0.7083
Train IoU: 0.5150 | Val IoU: 0.5542
Best model saved.


Epoch [2/20]: 100%|██████████| 170/170 [00:37<00:00,  4.48it/s, loss=0.279]



Epoch 2/20
Train Loss: 0.3289 | Val Loss: 0.2946
Train Dice: 0.7977 | Val Dice: 0.8177
Train IoU: 0.6657 | Val IoU: 0.6936
Best model saved.


Epoch [3/20]: 100%|██████████| 170/170 [00:37<00:00,  4.57it/s, loss=0.362]


...


Epoch 18/20
Train Loss: 0.1316 | Val Loss: 0.2032
Train Dice: 0.9169 | Val Dice: 0.8748
Train IoU: 0.8468 | Val IoU: 0.7783
Best model saved.


Epoch [19/20]: 100%|██████████| 170/170 [00:37<00:00,  4.59it/s, loss=0.15]



Epoch 19/20
Train Loss: 0.1250 | Val Loss: 0.2096
Train Dice: 0.9211 | Val Dice: 0.8717
Train IoU: 0.8540 | Val IoU: 0.7735
EarlyStopping Counter: 1/10


Epoch [20/20]: 100%|██████████| 170/170 [00:35<00:00,  4.83it/s, loss=0.134]



Epoch 20/20
Train Loss: 0.1267 | Val Loss: 0.2069
Train Dice: 0.9200 | Val Dice: 0.8727
Train IoU: 0.8522 | Val IoU: 0.7751
EarlyStopping Counter: 2/10


In [49]:
model.load_state_dict(torch.load(SAVE_PATH))
model.eval()

print("Best Swin-Unet model loaded successfully!")

Best Swin-Unet model loaded successfully!


In [50]:
def evaluate_model(model, loader):

    model.eval()

    total_loss = 0

    dice_scores = []
    iou_scores = []
    acc_scores = []
    precision_scores = []
    recall_scores = []
    f1_scores = []

    with torch.no_grad():

        for images, masks, _ in loader:

            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)

            loss = criterion(outputs, masks)

            total_loss += loss.item()

            dice, iou, acc, precision, recall, f1 = calculate_metrics(
                outputs,
                masks
            )

            dice_scores.append(dice)
            iou_scores.append(iou)
            acc_scores.append(acc)
            precision_scores.append(precision)
            recall_scores.append(recall)
            f1_scores.append(f1)

    return {
        "Loss": total_loss / len(loader),
        "Dice": np.mean(dice_scores),
        "IoU": np.mean(iou_scores),
        "Accuracy": np.mean(acc_scores),
        "Precision": np.mean(precision_scores),
        "Recall": np.mean(recall_scores),
        "F1": np.mean(f1_scores)
    }

In [51]:
train_metrics = evaluate_model(model, train_loader)
val_metrics = evaluate_model(model, val_loader)
test_metrics = evaluate_model(model, test_loader)

print("Evaluation Completed!")

Evaluation Completed!


In [52]:
import pandas as pd

results = pd.DataFrame({
    "Metric": [
        "Loss",
        "Dice",
        "IoU",
        "Accuracy",
        "Precision",
        "Recall",
        "F1"
    ],
    "Train": list(train_metrics.values()),
    "Validation": list(val_metrics.values()),
    "Test": list(test_metrics.values())
})

results

,Metric,Train,Validation,Test
0,Loss,0.110222,0.203192,0.201212
1,Dice,0.930427,0.874813,0.875877
2,IoU,0.870050,0.778298,0.780152
3,Accuracy,0.989160,0.980289,0.980803
4,Precision,0.935243,0.878453,0.890668
5,Recall,0.925851,0.873195,0.865188
6,F1,0.930427,0.874813,0.875877


In [53]:
import pandas as pd

def density_metrics(model, dataset, density_name):

    model.eval()

    density_results = []

    for density in [1, 2, 3, 4]:

        indices = np.where(dataset.labels == density)[0]

        dice_scores = []
        iou_scores = []

        with torch.no_grad():

            for idx in indices:

                image, mask, _ = dataset[idx]

                image = image.unsqueeze(0).to(device)
                mask = mask.unsqueeze(0).to(device)

                output = model(image)

                dice, iou, _, _, _, _ = calculate_metrics(
                    output,
                    mask
                )

                dice_scores.append(dice)
                iou_scores.append(iou)

        density_results.append({
            "Dataset": density_name,
            "Density": density,
            "Dice": np.mean(dice_scores),
            "IoU": np.mean(iou_scores)
        })

    return pd.DataFrame(density_results)

In [54]:
train_density = density_metrics(
    model,
    train_dataset,
    "Train"
)

val_density = density_metrics(
    model,
    val_dataset,
    "Validation"
)

test_density = density_metrics(
    model,
    test_dataset,
    "Test"
)

In [55]:
density_table = pd.concat(
    [
        train_density,
        val_density,
        test_density
    ],
    ignore_index=True
)

density_table

,Dataset,Density,Dice,IoU
0,Train,1,0.923413,0.859356
1,Train,2,0.927824,0.866490
2,Train,3,0.924014,0.860138
3,Train,4,0.919767,0.853121
4,Validation,1,0.885849,0.799587
5,Validation,2,0.879893,0.789145
6,Validation,3,0.864207,0.767317
7,Validation,4,0.870300,0.772444
8,Test,1,0.867575,0.770455
9,Test,2,0.881991,0.795303


In [56]:
pivot_table = density_table.pivot(
    index="Density",
    columns="Dataset",
    values=["Dice", "IoU"]
)

pivot_table = pivot_table.round(4)

pivot_table

Dice                        IoU                   
Dataset    Test   Train Validation    Test   Train Validation
Density                                                      
1        0.8676  0.9234     0.8858  0.7705  0.8594     0.7996
2        0.8820  0.9278     0.8799  0.7953  0.8665     0.7891
3        0.8780  0.9240     0.8642  0.7859  0.8601     0.7673
4        0.8486  0.9198     0.8703  0.7449  0.8531     0.7724

In [57]:
pivot_table = density_table.pivot(
    index="Density",
    columns="Dataset",
    values=["Dice", "IoU"]
)

pivot_table = pivot_table.round(4)

pivot_table

Dice                        IoU                   
Dataset    Test   Train Validation    Test   Train Validation
Density                                                      
1        0.8676  0.9234     0.8858  0.7705  0.8594     0.7996
2        0.8820  0.9278     0.8799  0.7953  0.8665     0.7891
3        0.8780  0.9240     0.8642  0.7859  0.8601     0.7673
4        0.8486  0.9198     0.8703  0.7449  0.8531     0.7724

In [58]:
import pandas as pd

density_table = pd.DataFrame({
    "Density": [1, 2, 3, 4],

    "Train Dice": train_density["Dice"].round(4),
    "Validation Dice": val_density["Dice"].round(4),
    "Test Dice": test_density["Dice"].round(4),

    "Train IoU": train_density["IoU"].round(4),
    "Validation IoU": val_density["IoU"].round(4),
    "Test IoU": test_density["IoU"].round(4)
})

density_table

,Density,Train Dice,Validation Dice,Test Dice,Train IoU,Validation IoU,Test IoU
0,1,0.9234,0.8858,0.8676,0.8594,0.7996,0.7705
1,2,0.9278,0.8799,0.8820,0.8665,0.7891,0.7953
2,3,0.9240,0.8642,0.8780,0.8601,0.7673,0.7859
3,4,0.9198,0.8703,0.8486,0.8531,0.7724,0.7449
